In [1]:
import pandas as pd

In [2]:
master_df = pd.read_csv("master_dataframe.csv")
master_df["date"] = pd.to_datetime(master_df["date"])
master_df.head()

,date,nf_Adj Close,nf_Close,nf_High,nf_Low,nf_Open,nf_Volume,bnf_Adj Close,bnf_Close,bnf_High,...,vix_level,vix_change,vix_5d_change,vix_ma_ratio,close_vs_252d_high,close_vs_252d_low,dow,ret_zscore,ma5_smooth_signal,volume_normalized
0,2022-01-03,17625.699219,17625.699219,17646.650391,17383.300781,17387.150391,200500,36421.476562,36421.898438,36492.1015625,...,16.450001,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.180301
1,2022-01-04,17805.250000,17805.250000,17827.599609,17593.550781,17681.400391,247400,36839.718750,36840.148438,36887.80078125,...,16.120001,-0.020061,NaN,NaN,NaN,NaN,1,1.144972,NaN,0.228022
2,2022-01-05,17925.250000,17925.250000,17944.699219,17748.849609,17820.099609,251500,37695.460938,37695.898438,37862.3984375,...,17.230000,0.068858,NaN,NaN,NaN,NaN,2,0.740212,0.008001,0.232194
3,2022-01-06,17745.900391,17745.900391,17797.949219,17655.550781,17768.500000,236500,37489.812500,37490.250000,37752.5,...,17.980000,0.043529,NaN,NaN,NaN,NaN,3,-1.225885,-0.006304,0.216931
4,2022-01-07,17812.699219,17812.699219,17905.000000,17704.550781,17797.599609,239300,37739.164062,37739.601562,38134.8515625,...,17.600000,-0.021135,NaN,NaN,NaN,NaN,4,0.390858,-0.005354,0.219780


In [3]:
selected_features = [
    "ret_1d",             # Core Momentum (Targets the Persistence Baseline)
    "ret_overnight",      # Sentiment (Captures the 'Dow' effect via the gap)
    "rsi_14",             # Mean Reversion (Exhaustion indicator)
    "close_vs_ma50",      # Long-term Trend (Regime anchor)
    "vix_change",         # Fear Flux (Daily sentiment shift)
    "vix_ma_ratio",       # Volatility Context (Identifies panic extremes)
    "bn_ret_1d",          # Cross-Asset Lead (Bank Nifty leadership)
    "nifty_bn_spread",    # Relative Strength (Sector rotation)
    "vol_20d",            # Risk Regime (Filters noise vs. trending vol)
    "ret_zscore",         # Statistical Outlier (Mean reversion trigger)
    "volume_ratio_20d",   # Conviction (Distinguishes 'Smart Money' moves)
    "close_vs_252d_high"  # Cycle Anchor (Replaces Dow: captures psychological resistance)
]

In [4]:
model_df = master_df[["date"] + selected_features + ["nf_Close"]].copy()    # 12 selected features + closing target(next day)
model_df.head()

,date,ret_1d,ret_overnight,rsi_14,close_vs_ma50,vix_change,vix_ma_ratio,bn_ret_1d,nifty_bn_spread,vol_20d,ret_zscore,volume_ratio_20d,close_vs_252d_high,nf_Close
0,2022-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17625.699219
1,2022-01-04,0.010187,0.003160,NaN,NaN,-0.020061,NaN,0.011483,-0.001297,NaN,1.144972,NaN,NaN,17805.250000
2,2022-01-05,0.006740,0.000834,NaN,NaN,0.068858,NaN,0.023229,-0.016489,NaN,0.740212,NaN,NaN,17925.250000
3,2022-01-06,-0.010005,-0.008745,NaN,NaN,0.043529,NaN,-0.005455,-0.004550,NaN,-1.225885,NaN,NaN,17745.900391
4,2022-01-07,0.003764,0.002913,NaN,NaN,-0.021135,NaN,0.006651,-0.002887,NaN,0.390858,NaN,NaN,17812.699219


In [ ]:
# target => 1-nxt day mkt close went up (wrt today's data)
# target => 0-nxt day mkt close went down (wrt today's data)
model_df["target"] = ( model_df["nf_Close"].shift(-1) > model_df["nf_Close"] ).astype(int)

# removed last row because Jan 1, 2026 data is unavailable, above code line gives NaN; .astype(int) converts to 0, which may not be true
model_df = model_df.iloc[:-1]

model_df.head()

In [ ]:
model_df = model_df.drop(columns=["nf_Close"])
display(model_df.head())
display(model_df.shape)

In [ ]:
# removing NaN

model_df = model_df.dropna()
display(model_df.head())
display(model_df.shape)

In [ ]:
# handling 0s (replacement for missing data, which may cause misdirection)
(model_df == 0).sum()
# zeros, where present are valid mathematically and logically

In [ ]:
model_df.to_csv("final_model_dataset.csv", index=False)
model_df.info()